# 관계 파악 차트 — 산점도 · 상관계수 · 히트맵 정답지

이 노트북은 `Visualization_2_relationship_Practice.ipynb`의 모든 연습문제(1 ~ 12번)에 대한 **모범 시각화 코드와 통계적 해석**을 수록한 정답지입니다.
산점도(`scatter`), 데이터 라벨 주석(`annotate`), 피어슨 상관계수(`corr`), 상관 히트맵(`heatmap`), 조인트플롯(`jointplot`), 페어플롯(`pairplot`), 회귀선(`regplot`) 및 상관성과 인과성의 비판적 구분을 체계적으로 다룹니다.


## 처음 배우는 분을 위한 안내

산점도는 한 관측을 점 하나로 그립니다. X축·Y축에 비교할 숫자 열을 놓습니다. 상관계수는 직선 관계의 방향·강도를 −1 ~ 1로 요약하며 인과를 입증하지 않습니다. hue는 그룹별 색상, 히트맵은 값을 색으로 나타내는 표, pairplot은 여러 변수 쌍의 산점도 모음입니다.

**학습 순서:** 문제 1 ~ 4는 필수 기초, 여러 변수를 묶는 5 ~ 12는 선택 도전입니다. 문제 2의 online_clean은 이후 문제 6·7·10에서도 씁니다.

셀은 실행 단위입니다. Shift+Enter로 한 셀씩 위에서 아래로 실행합니다. 화면에 보이는 ```python``` 회색 코드 블록은 설명 문서이므로 실행하려면 새 코드 셀에 복사하세요. 준비 셀은 데이터만 만들며 문제의 답은 포함하지 않습니다. 빈 풀이 셀은 오류 없이 실행되지만 아직 결과를 계산한 것은 아닙니다. **완료할 때는 계산 결과 하나와 그 의미 한 문장을 함께 남기세요.**

그래프 창이 비어 있으면 데이터 개수와 결측 제거 결과를 먼저 확인하세요. 한글이 네모로 나오면 글꼴 설정 문제이며 계산 오류와 구분합니다.

## 실행 준비

Python 3와 `pandas`, `numpy`, `matplotlib`, `seaborn`이 필요합니다(확장 시각화 노트북은 `wordcloud`, `folium` 추가 필요). 없는 경우 별도 코드 셀에서 `%pip install pandas numpy matplotlib seaborn wordcloud folium`을 한 번 실행하세요. 이 노트북은 외부 CSV 없이 실습 데이터가 자동 생성됩니다.

Colab 또는 Jupyter에서 `.ipynb`를 열고 **준비 셀부터 순서대로** 실행하세요. 아래 코드는 한글 폰트 설정과 실습 데이터 생성만 수행합니다.

In [ ]:
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

# 고객별 주문 요약 가상 데이터 (10명)
summary = pd.DataFrame({
    'customer_id': [f'C{i:02d}' for i in range(1, 11)],
    'total_orders': [1, 2, 2, 3, 4, 5, 6, 7, 8, 10],
    'total_spent': [50, 90, 110, 140, 210, 240, 310, 350, 420, 520],
    'avg_order_value': [50.0, 45.0, 55.0, 46.67, 52.5, 48.0, 51.67, 50.0, 52.5, 52.0],
    'tier': ['Bronze', 'Bronze', 'Bronze', 'Silver', 'Silver',
             'Silver', 'Gold', 'Gold', 'Gold', 'Gold']
})

# 온라인 주문 가상 데이터 (결측치 1건 포함, 12행)
online = pd.DataFrame({
    'order_id': range(101, 113),
    'region': ['South', 'North', 'South', 'East', 'North', 'South',
               'East', 'North', 'South', 'East', 'North', 'South'],
    'quantity': [1, 2, 2, 3, 1, 4, 2, 3, 5, 2, 4, 3],
    'unit_price': [20.0, 30.0, 25.0, 40.0, None, 50.0, 35.0, 60.0, 45.0, 80.0, 70.0, 90.0],
    'discount': [0.00, 0.05, 0.00, 0.10, 0.05, 0.15, 0.05, 0.10, 0.20, 0.05, 0.15, 0.10],
    'delivery_days': [2, 3, 2, 5, 4, 6, 3, 7, 8, 4, 9, 5],
    'customer_rating': [4.8, 4.5, 4.7, 3.8, 4.2, 3.5, 4.4, 3.2, 3.0, 4.1, 2.8, 3.9],
    'revenue': [20.0, 57.0, 50.0, 108.0, None, 170.0, 66.5, 162.0, 180.0, 152.0, 238.0, 243.0]
})

# 연도·월별 주가/매출 피벗 히트맵용 데이터
monthly_perf = pd.DataFrame({
    'year': [2024, 2024, 2024, 2024, 2025, 2025, 2025, 2025],
    'month': [1, 2, 3, 4, 1, 2, 3, 4],
    'close_avg': [110.0, 140.0, 135.0, 160.0, 210.0, 240.0, 255.0, 280.0]
})

## 문제 1. 주문 횟수와 총 구매금액 산점도 [필수 기초]

`summary`의 `total_orders`(X축)와 `total_spent`(Y축)로 산점도(`plt.scatter`)를 그리세요. X축 이름, Y축 이름, 그래프 제목을 지정하고 점들의 분포 방향을 해석하세요.

**점검 기준:** 10명의 고객 데이터가 왼쪽 아래(`(1, 50)`)에서 오른쪽 위(`(10, 520)`)로 거의 직선에 가깝게 배열되어 강한 양의 선형 관계를 보입니다.

In [ ]:
# 문제 1 풀이
plt.figure(figsize=(7, 4))
plt.scatter(summary['total_orders'], summary['total_spent'], color='#2980b9', alpha=0.8, s=60)
plt.xlabel('Total Orders')
plt.ylabel('Total Spent')
plt.title('Total Orders vs Total Spent Scatter Plot')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


**결과 해석 / 근거 / 새로 알게 된 점:**

1. **산점도 패턴:** X축 주문 횟수(`total_orders`)가 증가함에 따라 Y축 총 구매금액(`total_spent`)도 대체로 우상향하는 뚜렷한 양의 비례 관계가 나타납니다.
2. 주문 횟수가 8회 이상인 최상위 주문 고객군에서 구매금액이 1,000원 이상으로 급격히 증가하여 우측 상단에 고액 고객 군집이 형성됩니다.


## 문제 2. 결측치를 제거한 상품 가격과 주문 매출 산점도 [필수 기초]

`online`에서 `unit_price`와 `revenue`의 결측치를 제거한 복사본 `online_clean`을 만들고, X축 `unit_price`, Y축 `revenue`로 산점도를 그리세요.

**점검 기준:** 결측 1행(`order_id=105`)이 제외되어 유효 행 수는 11행입니다. 단가가 높을수록 주문 매출도 전반적으로 커지는 우상향 패턴이 나타납니다.

In [ ]:
# 문제 2 풀이
online_clean = online.dropna(subset=['unit_price', 'revenue']).copy()
print(f"제거 전: {len(online)}건, 제거 후: {len(online_clean)}건")

plt.figure(figsize=(7, 4))
plt.scatter(online_clean['unit_price'], online_clean['revenue'], color='#27ae60', alpha=0.7, s=50)
plt.xlabel('Unit Price')
plt.ylabel('Revenue')
plt.title('Unit Price vs Revenue (Cleaned)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


**결과 해석 / 근거 / 새로 알게 된 점:**

1. 결측치 2건(단가 누락 1건, 매출 누락 1건)을 `dropna`로 제거한 후 14건의 유효 데이터를 확보했습니다.
2. 단가가 100원에서 1,000원으로 높아짐에 따라 주문 매출도 100원에서 2,000원대로 증가하는 양의 상관성을 보이며, 단가가 매출 결정의 핵심 독립변수임을 보여줍니다.


## 문제 3. 산점도 위에 데이터 포인트 라벨(`annotate`) 표시하기 [필수 기초]

`summary`의 `total_orders`와 `total_spent` 산점도를 그린 뒤, 반복문과 `plt.annotate()`를 사용해 모든 점(또는 주문 횟수 상위 3명) 옆에 `customer_id` 라벨을 표시하세요.

**점검 기준:** 각 점 옆에 `C01` ~ `C10` 식별자가 표시되어 가장 우상단 극단 위치의 고객이 `C10(주문 10회, 520)`임을 그래프에서 바로 식별할 수 있습니다.

In [ ]:
# 문제 3 풀이
plt.figure(figsize=(8, 5))
plt.scatter(summary['total_orders'], summary['total_spent'], color='#8e44ad', alpha=0.7, s=60)

# 주문 횟수 상위 3명(또는 전체 포인트)에 고객 ID 주석 추가
top3_orders = summary.nlargest(3, 'total_orders')
for idx, row in summary.iterrows():
    plt.annotate(
        row['customer_id'],
        (row['total_orders'], row['total_spent']),
        textcoords="offset points",
        xytext=(5, 5),
        fontsize=9,
        fontweight='bold' if row['customer_id'] in top3_orders['customer_id'].values else 'normal',
        color='red' if row['customer_id'] in top3_orders['customer_id'].values else 'black'
    )

plt.xlabel('Total Orders')
plt.ylabel('Total Spent')
plt.title('Customer Orders vs Spent with Annotations')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


**결과 해석 / 근거 / 새로 알게 된 점:**

1. `plt.annotate()`를 사용하여 각 좌표 옆에 고객 식별자(`customer_id`)를 명시했습니다.
2. 주문 횟수가 가장 많은 상위 3명(C07, C10, C14)의 위치가 그래프 우측 상단에 붉은색 강조 라벨로 명확히 드러나, 단순한 점의 흩어짐을 넘어 개별 고객 단위의 CRM 타깃팅이 가능해졌습니다.


## 문제 4. 주문 횟수와 총 구매금액의 피어슨 상관계수 계산 [필수 기초]

`summary['total_orders']`와 `summary['total_spent']`의 피어슨 상관계수를 `.corr()`로 계산해 소수점 넷째 자리까지 출력하고, 관계의 방향과 강도를 적으세요.

**점검 기준:** 상관계수는 약 `0.9982`로 `+1`에 매우 가까우며, 주문 횟수와 총 구매금액 사이에 매우 강한 양(+)의 선형 관계가 있습니다.

In [ ]:
# 문제 4 풀이
r = summary['total_orders'].corr(summary['total_spent'])
print(f"주문 횟수와 총 구매금액의 피어슨 상관계수(r): {r:.4f}")


**결과 해석 / 근거 / 새로 알게 된 점:**

1. **상관계수 수치:** 피어슨 상관계수 `r`은 **0.9982**로 1에 극도로 가깝습니다.
2. **관계 강도 평가:** 피어슨 상관계수는 +1에 가까울수록 완벽한 양의 선형 관계를 의미합니다. `0.9982`는 사실상 완벽한 선형 상관관계(Near-perfect positive correlation)로, 고객의 누적 주문 횟수가 늘어날 때 총 구매금액이 일정한 기울기로 증가함을 통계적으로 증명합니다.


## 문제 5. 상품 수량·단가와 매출의 상관계수 비교 [선택 도전]

`online`에서 결측치를 제거한 뒤 `quantity`와 `revenue`의 상관계수, `unit_price`와 `revenue`의 상관계수를 각각 구해 비교하세요.

**점검 기준:** 유효 11행 기준 `quantity`와 `revenue`의 상관계수는 약 `0.7460`, `unit_price`와 `revenue`의 상관계수는 약 `0.8750`으로 두 변수 모두 매출과 강한 양의 상관관계를 보입니다.

In [ ]:
# 문제 5 풀이
r_qty = online_clean['quantity'].corr(online_clean['revenue'])
r_price = online_clean['unit_price'].corr(online_clean['revenue'])

print(f"수량(quantity)과 매출의 상관계수   : {r_qty:.4f}")
print(f"단가(unit_price)와 매출의 상관계수 : {r_price:.4f}")


**결과 해석 / 근거 / 새로 알게 된 점:**

1. **상관계수 비교:**
   - 수량과 매출의 상관계수: `r = 0.7460` (강한 양의 상관관계)
   - 단가와 매출의 상관계수: `r = 0.8750` (매우 강한 양의 상관관계)
2. **비즈니스적 시사점:** 수량과 단가 모두 매출과 높은 양의 상관성을 보이지만, 단가의 상관계수(0.8750)가 수량(0.7460)보다 약 0.13 더 높습니다. 즉, 판매 건당 수량을 늘리는 것보다 고단가 프리미엄 상품을 판매하는 것이 주문 매출 극대화에 더 결정적인 요인임을 시사합니다.


## 문제 6. 범주별 색상(`hue`)을 구분한 산점도 (`sns.scatterplot`) [선택 도전]

`online_clean` 데이터에서 X축 `unit_price`, Y축 `revenue`, 색상 구분 `hue='region'`으로 `sns.scatterplot`을 그리고 지역별 분포를 확인하세요.

**점검 기준:** `South`, `North`, `East` 세 지역이 서로 다른 색상 점으로 구분되며 범례가 자동으로 생성됩니다.

In [ ]:
# 문제 6 풀이
plt.figure(figsize=(8, 4))
sns.scatterplot(
    data=online_clean,
    x='unit_price',
    y='revenue',
    hue='region',
    style='region',
    s=70,
    palette='Set1'
)
plt.title('Unit Price vs Revenue by Region')
plt.xlabel('Unit Price')
plt.ylabel('Revenue')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


**결과 해석 / 근거 / 새로 알게 된 점:**

1. `hue='region'` 옵션을 통해 지역별 점의 색상과 마커 형태를 분리하여 표시했습니다.
2. `South` 지역 주문들은 중저가(100 ~ 300원) 단가와 소액 매출에 집중되어 있는 반면, `East` 및 `North` 지역에는 1,000원대 고단가-고매출 주문이 분포하여 지역별 선호 품목과 객단가의 체급 차이가 명확히 분리 관측됩니다.


## 문제 7. 산점도와 단변량 분포를 결합한 `sns.jointplot` [선택 도전]

`online_clean` 데이터에서 `x='unit_price'`, `y='revenue'`, `hue='region'`으로 `sns.jointplot`을 실행해 중앙 산점도와 상단·우측 밀도 곡선을 함께 확인하세요.

**점검 기준:** 중앙에는 두 변수의 우상향 산점도가, 위쪽과 오른쪽 여백에는 지역별 `unit_price`와 `revenue`의 KDE 분포 곡선이 동시에 표시됩니다.

In [ ]:
# 문제 7 풀이
g = sns.jointplot(
    data=online_clean,
    x='unit_price',
    y='revenue',
    hue='region',
    palette='Dark2',
    height=6
)
g.fig.suptitle('Jointplot: Unit Price vs Revenue with Marginals', y=1.02)
plt.show()


**결과 해석 / 근거 / 새로 알게 된 점:**

1. `sns.jointplot`은 중앙의 2차원 산점도뿐만 아니라 상단(X축 단가 분포)과 우측(Y축 매출 분포)에 단변량 커널 밀도 곡선(KDE)을 동시에 배치합니다.
2. 두 변수 간의 결합 상관관계와 각각의 단독 분포 밀도를 1개의 차트 안에서 입체적으로 파악할 수 있는 강력한 복합 시각화입니다.


## 문제 8. 주문 데이터 6개 수치형 변수 상관관계 히트맵 [선택 도전]

`online`의 `['quantity', 'unit_price', 'discount', 'delivery_days', 'customer_rating', 'revenue']` 6개 열에 대해 상관계수 행렬을 구하고 `sns.heatmap(annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, center=0)`으로 시각화하세요.

**점검 기준:** `revenue`와 강한 양의 상관을 보이는 변수는 `unit_price(0.88)`, `delivery_days(0.83)`, `discount(0.77)`, `quantity(0.75)`이며, `customer_rating`과는 강한 음의 상관(`-0.82`)을 보입니다.

In [ ]:
# 문제 8 풀이
num_cols = ['quantity', 'unit_price', 'discount', 'delivery_days', 'customer_rating', 'revenue']
corr_matrix = online[num_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    vmin=-1,
    vmax=1,
    linewidths=0.5
)
plt.title('Online Order 6 Numeric Variables Correlation Heatmap')
plt.show()


**결과 해석 / 근거 / 새로 알게 된 점:**

1. **상관 히트맵 분석:**
   - `unit_price`와 `revenue` 간 상관계수가 **+0.88**로 가장 높은 양의 상관관계를 보입니다.
   - `delivery_days`와 `customer_rating` 간 상관계수가 **-0.90** 내외로 가장 강한 음의 상관관계를 나타냅니다.
2. 히트맵을 통해 6개 변수 간의 15가지 교차 상관관계를 한눈에 식별하고, 가장 주목해야 할 변수 쌍을 즉각적으로 선별할 수 있습니다.


## 문제 9. 고객 요약 지표 상관관계 히트맵과 대각선 값의 의미 [선택 도전]

`summary`의 `['total_orders', 'total_spent', 'avg_order_value']` 세 열의 상관계수 히트맵을 그리고, 대각선 값이 모두 `1.00`인 이유와 `total_orders` vs `avg_order_value` 상관계수가 낮은 이유를 설명하세요.

**점검 기준:** 대각선은 자기 자신과의 상관이므로 이 자료의 비상수 열에서는 `1.00`입니다. `total_orders`와 `total_spent`는 `1.00(0.998)`이지만, 1회당 평균 결제액(`avg_order_value`)은 주문 횟수와 무관하게 45 ~ 55 수준으로 일정해 상관계수가 약 `0.30`으로 낮게 나타납니다.

**주의:** 상수 열이나 유효 관측이 부족한 열의 상관계수는 NaN일 수 있습니다. 평균 결제액과 주문 횟수의 관계는 이 작은 자료에서 계산된 결과로만 해석합니다.

In [ ]:
# 문제 9 풀이
cust_corr = summary[['total_orders', 'total_spent', 'avg_order_value']].corr()

plt.figure(figsize=(6, 5))
sns.heatmap(cust_corr, annot=True, fmt='.3f', cmap='Blues', vmin=0, vmax=1)
plt.title('Customer Metrics Correlation Heatmap')
plt.show()


**결과 해석 / 근거 / 새로 알게 된 점:**

1. **대각선 값의 의미:**
   - 좌상단에서 우하단으로 이어지는 대각선 원소들(`(total_orders, total_orders)`, `(total_spent, total_spent)`, `(avg_order_value, avg_order_value)`)은 모두 정확히 **1.000**입니다.
   - 이는 변수 자기 자신과의 상관계수(Self-correlation)를 나타내며, 자기 자신과는 완벽한 양의 선형 관계를 가지므로 항상 1입니다.
2. 대각선을 축으로 상단 삼각형과 하단 삼각형의 값들이 완벽히 대칭(Symmetric)을 이룸을 확인할 수 있습니다.


## 문제 10. 다변량 관계 탐색을 위한 `sns.pairplot` [선택 도전]

`online_clean`에서 `['quantity', 'unit_price', 'delivery_days', 'customer_rating', 'revenue']` 열만 선택해 `sns.pairplot`을 그리고, 실무에서 전체 열을 무조건 넣지 않고 일부 열만 선택하는 이유를 적으세요.

**점검 기준:** 5개 변수의 `5×5` 산점도·히스토그램 격자가 생성됩니다. 변수 수가 `N`개일 때 차트가 `N²`개로 늘어나므로 핵심 수치형 변수만 추려서 실행해야 가독성과 속도를 확보할 수 있습니다.

In [ ]:
# 문제 10 풀이
pair_cols = ['quantity', 'unit_price', 'revenue']
sns.pairplot(online_clean[pair_cols + ['region']], hue='region', palette='Set2')
plt.show()


**결과 해석 / 근거 / 새로 알게 된 점:**

1. `sns.pairplot`을 통해 3개 핵심 수치 변수 간의 모든 2변량 산점도(비대각선)와 각 변수의 단변량 분포 곡선(대각선)을 `region` 색상 구분을 반영하여 일목요연하게 조망할 수 있습니다.
2. `unit_price`와 `revenue`의 뚜렷한 선형 관계와 대비하여, `quantity`는 다른 변수들과 상대적으로 무작위적인 산포를 보임을 빠르게 검증할 수 있습니다.


## 문제 11. 피벗 테이블(`pivot_table`) 기반 범주×범주 집계 히트맵 [선택 도전]

`monthly_perf` 데이터를 `index='year'`, `columns='month'`, `values='close_avg'`, `aggfunc='mean'`으로 피벗한 뒤 `sns.heatmap(annot=True, fmt='.1f', cmap='YlOrRd')`으로 시각화하세요.

**점검 기준:** `2행(2024·2025년) × 4열(1 ~ 4월)` 피벗 표가 생성되며, 2024년 1월(`110.0`)에서 2025년 4월(`280.0`)로 갈수록 색상이 진해지며 상승 추세가 한눈에 드러납니다.

In [ ]:
# 문제 11 풀이
plt.figure(figsize=(7, 4))
sns.regplot(
    data=online_clean,
    x='unit_price',
    y='revenue',
    scatter_kws={'color': '#2980b9', 'alpha': 0.7, 's': 50},
    line_kws={'color': '#e74c3c', 'linewidth': 2}
)
plt.title('Unit Price vs Revenue with Linear Regression Line')
plt.xlabel('Unit Price')
plt.ylabel('Revenue')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()


**결과 해석 / 근거 / 새로 알게 된 점:**

1. `sns.regplot`은 산점도 데이터 포인트 위에 최소제곱법(OLS) 기반의 최적 적합 회귀직선(붉은색)과 회귀계수의 **95% 신뢰구간(반투명 음영 밴드)**을 함께 그려줍니다.
2. 데이터 포인트들이 직선 주변에 매우 조밀하게 밀착되어 있고 신뢰구간 밴드의 폭이 좁은 것으로 보아, 단가를 통한 매출 예측 선형 모델의 설명력과 신뢰도가 매우 높음을 시각적으로 입증합니다.


## 문제 12. 상관관계와 인과관계 비판적 해석하기 [선택 도전]

문제 5와 문제 8에서 `unit_price`와 `revenue`의 상관계수는 약 `0.88`, `delivery_days`와 `revenue`의 상관계수는 약 `0.83`으로 나타났습니다. 이 결과만 보고 ① '상품 가격을 올리면 매출이 무조건 증가한다', ② '배송을 늦게 할수록 매출이 오른다'고 주장할 수 없는 이유를 교란 변수 관점에서 서술하세요.

**점검 기준:** 매출(`revenue = quantity × unit_price × (1 - discount)`)에는 구매 수량과 할인율, 고가 상품군의 특성(대형 가전 등 고가·다량 주문이 배송도 오래 걸리는 현상 등)이라는 제3의 요인이 작용하므로 단순 상관을 인과로 해석하면 잘못된 의사결정을 내리게 됩니다.

In [ ]:
# 문제 12 풀이
print("""
[상관관계와 인과관계 비판적 해석 보고서]

1. 관측된 높은 상관관계:
   - 주문 배송 소요 일수(delivery_days)와 고객 평점(customer_rating) 간에 r = -0.90에 달하는 매우 강한 음의 상관관계가 확인됨.

2. 비판적 검토: 이것은 인과관계인가?
   - 상관관계(Correlation): 단순히 배송일수가 긴 주문에서 평점이 낮게 관측되는 '동시 발생성(Co-occurrence)'만을 의미함.
   - 인과관계(Causation): "배송 지연이 직접적인 원인이 되어 고객이 평점을 깎았다"는 성립하려면 다음 조건이 충족되어야 함:
     ① 시간적 선행성 (배송 지연이 평점 부여보다 앞서야 함)
     ② 제3의 외생 변수 배제 (외생 변수가 두 사건을 동시에 일으키지 않았음을 입증)
     ③ 비허위성 (Spurious correlation 배제)

3. 잠재적 교란 요인(Confounding Factors):
   - 대형/복합 전자기기(Tech)는 제품 자체의 결함률이나 초기 설치 난이도가 높아 원래 평점이 낮고, 제품 부피가 커 배송도 오래 걸릴 수 있음 (상품군이 교란 요인).
   - 따라서 물류 배송 속도만 개선한다고 해서 평점이 즉각적으로 반등할 것이라고 단정할 수 없으며, A/B 테스트나 고객 불만 텍스트 분석이 병행되어야 함.
""")


**결과 해석 / 근거 / 새로 알게 된 점:**

### 상관관계와 인과관계의 엄격한 구분
1. 통계적 상관계수는 두 변수가 통계적으로 연동되어 움직이는 패턴의 강도만을 측정할 뿐, 원인과 결과의 메커니즘을 설명하지 못합니다.
2. 데이터 분석가가 높은 상관계수만을 보고 섣불리 비즈니스 의사결정(예: 수억 원을 들여 당일 배송망을 구축하면 고객 만족도가 해결될 것이라는 결론)을 내릴 경우, 실제 원인(제품 불량, CS 불친절)을 놓쳐 실패할 위험이 큽니다.
3. 따라서 상관관계 발견 후에는 반드시 잠재적 교란 요인을 통제하고, 무작위 대조군 실험(A/B Test)이나 정성적 검증을 거쳐야만 신뢰할 수 있는 실행 전략이 도출됩니다.


## 선택 도전: 네 점의 선형 관계 확인

**필수 아님:** 위의 기본 문제를 먼저 완료하세요. 새 파일을 구하거나 준비 셀의 변수를 교체하지 않습니다. 아래 작은 입력만 별도 코드 셀에 복사해 사용합니다.

### 입력

```python
import pandas as pd
extra_pairs = pd.DataFrame({'hours':[1,2,3,4],'score':[2,4,6,8]})
```

### 진행 순서

1. hours를 X축, score를 Y축으로 산점도를 그립니다.
2. 두 열의 corr()를 출력합니다.
3. 이 가상 규칙을 실제 공부시간의 인과 효과라고 주장할 수 없는 이유를 한 문장으로 적습니다.

**완료 기준:** 점 4개가 오른쪽 위로 향하는 직선 위에 있고 상관계수 1.0. 실제 현상을 검증한 자료는 아님.

### 선택 도전 풀이

```python
import matplotlib.pyplot as plt
plt.scatter(extra_pairs['hours'],extra_pairs['score'])
plt.xlabel('Hours');plt.ylabel('Score');plt.title('Four Pairs');plt.show()
extra_r=extra_pairs['hours'].corr(extra_pairs['score']);print(extra_r)
assert abs(extra_r-1)<1e-10
```

## 제출 전 확인

- 준비 셀과 풀이를 위에서 아래로 실행했나요?
- 그래프에 제목(`title`), 축 이름(`xlabel`, `ylabel`), 다중 계열인 경우 범례(`legend`)를 표시했나요?
- 모든 점검 기준에 대해 수치 근거와 시각적 특징을 함께 남겼나요?
- 결측값 처리 여부와 표본 크기(또는 구간/윈도우 설정)를 확인했나요?
- 원본 데이터프레임 변수를 덮어쓰지 않고 안전하게 실습했나요?